# Retroviral Wall Challenge — Final Solution

**CLS 0.6936** (nested LOFO CV)

Blend of 3 models:
1. ElasticNet on 35 tabular features (biophysical + zero-shot ESM2)
2. Ridge on PCA3 of ESM2 layer 12 mid-region
3. Ridge on PCA3 of ESM2 layer 33 mid-region

Weights optimized by nested LOFO (inner LOFO per outer fold).

In [ ]:
import os, warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from itertools import product as iprod
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import average_precision_score
from transformers import EsmTokenizer, EsmModel

## 1. Load data

In [ ]:
DATA_DIR = '../data/raw'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
splits_df = pd.read_csv(f'{DATA_DIR}/family_splits.csv')
splits = {}
for _, row in splits_df.iterrows():
    splits[row['family']] = row['rt_names'].split('|')

gt_order = pd.read_csv(f'{DATA_DIR}/rt_sequences.csv', usecols=['rt_name'])['rt_name'].tolist()
sequences = dict(zip(train['rt_name'], train['sequence']))
families = list(splits.keys())

print(f'{len(train)} RTs, {len(families)} families')

## 2. Feature engineering

In [ ]:
# Missing flags
for col in ['connection_mean_pot', 'triad_best_rmsd', 'D1_D2_dist', 'D2_D3_dist',
            'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'yxdd_5A_mean_pot']:
    train[f'{col}_missing'] = train[col].isna().astype(int)

# Interactions
train['foldseek_gap_MMLV'] = train['foldseek_best_TM'] - train['foldseek_TM_MMLV']
train['t40_x_foldseek_MMLV'] = train['t40_raw'] * train['foldseek_TM_MMLV']
train['triad_quality'] = train['triad_found_bin'] * (1 / (train['triad_best_rmsd'].fillna(99) + 1))
train['seq_struct_compat'] = -train['perplexity'] * train['instability_index']

## 3. ESM2 zero-shot features + layer 12/33 mid-region embeddings

In [ ]:
print('Loading ESM2-650M...')
tokenizer = EsmTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
model = EsmModel.from_pretrained('facebook/esm2_t33_650M_UR50D')
model.eval()

l12_mid, l33_mid = {}, {}
for rt_name, seq in sequences.items():
    inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        n_third = len(seq) // 3
        l12_mid[rt_name] = outputs.hidden_states[12][0, 1:-1, :][n_third:2*n_third].mean(dim=0).numpy()
        l33_mid[rt_name] = outputs.hidden_states[33][0, 1:-1, :][n_third:2*n_third].mean(dim=0).numpy()

del model
print('Layer embeddings extracted.')

# Zero-shot features (pseudo-perplexity, entropy)
from transformers import EsmForMaskedLM
model_mlm = EsmForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D')
model_mlm.eval()

for rt_name, seq in sequences.items():
    inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024)
    with torch.no_grad():
        logits = model_mlm(**inputs).logits
        probs = torch.softmax(logits[0, 1:-1, :], dim=-1)
        input_ids = inputs['input_ids'][0, 1:-1]
        per_pos_ll = torch.log(probs[range(len(input_ids)), input_ids])
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
    train.loc[train['rt_name'] == rt_name, 'esm2_pseudo_ppl'] = torch.exp(-per_pos_ll.mean()).item()
    train.loc[train['rt_name'] == rt_name, 'esm2_mean_entropy'] = entropy.mean().item()
    train.loc[train['rt_name'] == rt_name, 'esm2_mean_ll'] = per_pos_ll.mean().item()

del model_mlm
print('Zero-shot features extracted.')

# PCA on mid-region embeddings
l12_arr = np.array([l12_mid[n] for n in train['rt_name']])
l33_arr = np.array([l33_mid[n] for n in train['rt_name']])
pca12 = PCA(n_components=3).fit(l12_arr)
pca33 = PCA(n_components=3).fit(l33_arr)

L12_COLS = [f'l12_pc{i}' for i in range(3)]
L33_COLS = [f'l33_pc{i}' for i in range(3)]
for i in range(3):
    train[L12_COLS[i]] = pca12.transform(l12_arr)[:, i]
    train[L33_COLS[i]] = pca33.transform(l33_arr)[:, i]

print('PCA done.')

## 4. Define feature sets and models

In [ ]:
FEATURES_BASE = [c for c in [
    'foldseek_TM_MMLV', 'foldseek_TM_MMLVPE', 'foldseek_best_TM',
    'foldseek_best_LDDT', 'foldseek_best_fident', 'foldseek_TM_HIV1',
    'triad_found_bin', 'triad_best_rmsd', 'perplexity', 'log_likelihood',
    'D1_D2_dist', 'D2_D3_dist',
    't40_raw', 't45_raw', 't50_raw', 't55_raw', 't60_raw',
    'instability_index', 'gravy', 'camsol', 'net_charge',
] if c in train.columns] + [
    f'{c}_missing' for c in ['connection_mean_pot', 'triad_best_rmsd', 'D1_D2_dist',
    'D2_D3_dist', 'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'yxdd_5A_mean_pot']
] + ['foldseek_gap_MMLV', 't40_x_foldseek_MMLV', 'triad_quality', 'seq_struct_compat',
     'esm2_pseudo_ppl', 'esm2_mean_entropy', 'esm2_mean_ll']

print(f'Tabular features: {len(FEATURES_BASE)}')

def make_pipe(est):
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), est)

def normalize(s):
    mn, mx = s.min(), s.max()
    if mx - mn < 1e-12: return np.zeros_like(s, dtype=float)
    return (s - mn) / (mx - mn)

def en_fn(Xtr, ytr, Xte):
    return make_pipe(ElasticNet(alpha=1.0, l1_ratio=0.3, max_iter=10000)).fit(Xtr, ytr).predict(Xte)

def ridge_fn(Xtr, ytr, Xte):
    return make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(Xtr, ytr).predict(Xte)

MODELS = [
    ('EN', en_fn, FEATURES_BASE),
    ('L12', ridge_fn, L12_COLS),
    ('L33', ridge_fn, L33_COLS),
]

## 5. Nested LOFO CV + blend

In [ ]:
all_oof = []

for outer_family in families:
    outer_rts = splits[outer_family]
    outer_mask = train['rt_name'].isin(outer_rts)
    inner_mask = ~outer_mask
    inner_splits = {f: rts for f, rts in splits.items() if f != outer_family}
    inner_df = train[inner_mask].reset_index(drop=True)

    # Inner LOFO: generate OOF for each model
    inner_oof = {}
    for mname, fn, mfc in MODELS:
        oof_vals = np.full(len(inner_df), np.nan)
        for ifam, irts in inner_splits.items():
            tm = inner_df['rt_name'].isin(irts)
            trm = ~tm
            preds = fn(inner_df.loc[trm, mfc].values, inner_df.loc[trm, 'pe_efficiency_pct'].values,
                      inner_df.loc[tm, mfc].values)
            oof_vals[tm.values] = preds
        inner_oof[mname] = oof_vals

    # Find best weights on inner OOF
    model_names = [m[0] for m in MODELS]
    best_cls, best_w = -1, None
    for weights in iprod(np.arange(0, 1.05, 0.1), repeat=3):
        if abs(sum(weights) - 1.0) > 0.01: continue
        b = sum(w * normalize(inner_oof[n]) for w, n in zip(weights, model_names))
        # Inline CLS computation
        y_true = inner_df['active'].values
        pe_eff = inner_df['pe_efficiency_pct'].values
        pr_auc = average_precision_score(y_true, b)
        # Weighted Spearman
        pred_ranks = np.argsort(np.argsort(b)).astype(float)
        true_ranks = np.argsort(np.argsort(pe_eff)).astype(float)
        w_arr = (pe_eff + 0.01); w_arr = w_arr / w_arr.sum()
        mu_p, mu_t = w_arr @ pred_ranks, w_arr @ true_ranks
        dp, dt = pred_ranks - mu_p, true_ranks - mu_t
        cov = (w_arr * dp * dt).sum()
        std_p, std_t = np.sqrt((w_arr * dp**2).sum()), np.sqrt((w_arr * dt**2).sum())
        w_sp = max(cov / (std_p * std_t) if std_p > 1e-12 and std_t > 1e-12 else 0.0, 0.0)
        cls = 2 * pr_auc * w_sp / (pr_auc + w_sp) if pr_auc > 0 and w_sp > 0 else 0.0
        if cls > best_cls:
            best_cls = cls
            best_w = dict(zip(model_names, weights))

    # Predict outer fold
    blended = np.zeros(outer_mask.sum())
    for mname, fn, mfc in MODELS:
        pred = fn(train.loc[inner_mask, mfc].values, train.loc[inner_mask, 'pe_efficiency_pct'].values,
                 train.loc[outer_mask, mfc].values)
        mn, mx = inner_oof[mname].min(), inner_oof[mname].max()
        normed = (pred - mn) / (mx - mn) if mx - mn > 1e-12 else np.zeros_like(pred)
        blended += best_w[mname] * normed

    fold_df = train.loc[outer_mask, ['rt_name', 'active', 'pe_efficiency_pct', 'rt_family']].copy()
    fold_df['predicted_score'] = blended
    all_oof.append(fold_df)

    print(f'  {outer_family:<25s} weights: EN={best_w["EN"]:.1f} L12={best_w["L12"]:.1f} L33={best_w["L33"]:.1f}')

oof = pd.concat(all_oof).set_index('rt_name').loc[gt_order].reset_index()
print(f'\nAll 57 OOF predictions generated.')

## 6. Score and save submission

In [ ]:
# Compute CLS
y_true = oof['active'].values
y_score = oof['predicted_score'].values
pe_eff = oof['pe_efficiency_pct'].values

pr_auc = average_precision_score(y_true, y_score)
pred_ranks = np.argsort(np.argsort(y_score)).astype(float)
true_ranks = np.argsort(np.argsort(pe_eff)).astype(float)
w = (pe_eff + 0.01); w = w / w.sum()
mu_p, mu_t = w @ pred_ranks, w @ true_ranks
dp, dt = pred_ranks - mu_p, true_ranks - mu_t
cov = (w * dp * dt).sum()
std_p, std_t = np.sqrt((w * dp**2).sum()), np.sqrt((w * dt**2).sum())
w_sp = max(cov / (std_p * std_t) if std_p > 1e-12 and std_t > 1e-12 else 0.0, 0.0)
cls = 2 * pr_auc * w_sp / (pr_auc + w_sp) if pr_auc > 0 and w_sp > 0 else 0.0

print(f'PR-AUC:            {pr_auc:.4f}')
print(f'Weighted Spearman: {w_sp:.4f}')
print(f'CLS:               {cls:.4f}')

# Save submission
submission = oof[['rt_name', 'predicted_score']].copy()
submission['predicted_active'] = (submission['predicted_score'] > submission['predicted_score'].median()).astype(int)
submission = submission[['rt_name', 'predicted_active', 'predicted_score']]
submission.to_csv('submission.csv', index=False)
print(f'\nSubmission saved: {len(submission)} rows')